# Introduction to FastAPI

## FastAPI

FastAPI is a modern, high-performance Python web framework for building APIs with automatic OpenAPI documentation, type checking, and asynchronous support. It is designed for speed, efficiency, and developer productivity, particularly for RESTful APIs and microservices.

## FastAPI over Flask or Django

We chose FastAPI because our application is an AI-powered RAG chatbot that makes frequent I/O-bound calls to services like Azure OpenAI, Azure AI Search, Redis, and Cosmos DB. FastAPI supports asynchronous programming with async and await, allowing it to handle many concurrent requests efficiently while waiting for these external services. It also provides automatic request validation using Pydantic, built-in OpenAPI/Swagger documentation, dependency injection, and excellent performance through ASGI. Compared to Flask, FastAPI requires less boilerplate and has native async support. Compared to Django, it is more lightweight and API-focused, making it a better fit for microservices and AI applications.


## ASGI

ASGI (Asynchronous Server Gateway Interface) is the standard interface between Python web applications and web servers. It is the successor to WSGI and is designed to support asynchronous programming.

It enables Python frameworks like FastAPI, Starlette, and Django (ASGI mode) to handle many concurrent requests efficiently.

## ASGI and WSGI

WSGI is the traditional Python interface between web servers and applications. It supports only synchronous request handling, so each request blocks its worker until processing is complete. 

ASGI is the modern successor that supports both synchronous and asynchronous execution. 
With ASGI, applications can use async and await, enabling high concurrency, WebSockets, Server-Sent Events, and streaming responses. Frameworks like FastAPI use ASGI with servers such as Uvicorn, making them well-suited for AI applications that frequently wait on databases, Redis, or LLM APIs.

## Uvicron 

Uvicorn is a lightweight, high-performance ASGI web server used to run asynchronous Python web applications such as FastAPI and Starlette.

It acts as the bridge between the client (browser or API consumer) and your FastAPI application.



## API

An API (Application Programming Interface) is a set of rules that allows two software systems to communicate with each other — it acts like a “messenger” between applications, enabling data exchange and functionality sharing.

## REST API

A REST API (Representational State Transfer API) is a standard way for clients (apps, browsers, services) to communicate with servers over HTTP, using simple methods like GET, POST, PUT, PATCH, and DELETE to perform CRUD operations (Create, Read, Update, Delete). It’s widely used because it’s lightweight, scalable, and easy to integrate.

### Why REST 
It’s simple, universal, and easy to learn — ideal for both beginners and large‑scale applications.

### Status Codes 

| HTTP Code                     | Meaning / When to Use in RAG API                                 | Common Error / Example                                        |
| ----------------------------- | ---------------------------------------------------------------- | ------------------------------------------------------------- |
| **200 OK**                    | Successful retrieval of documents or LLM-generated answer        | Usually none; normal response                                 |
| **201 Created**               | When a new resource is added (e.g., new document indexed)        | If POST fails to store the document → 500/422                 |
| **204 No Content**            | Successful request with no response body (e.g., delete document) | Misused if client expects data back                           |
| **400 Bad Request**           | Invalid query format, missing required parameters, wrong JSON    | Malformed query JSON, missing `query` field                   |
| **401 Unauthorized**          | Authentication missing or invalid (e.g., API key)                | Calling RAG API without token or expired key                  |
| **403 Forbidden**             | Authenticated but no permission to access resource               | User not allowed to access certain documents                  |
| **404 Not Found**             | Document, chunk, or resource does not exist                      | Querying non-existent document ID or page chunk               |
| **409 Conflict**              | Resource already exists (e.g., duplicate document)               | Attempting to index a PDF with same ID twice                  |
| **422 Unprocessable Entity**  | Input validation failed                                          | Invalid data type for query or embedding request              |
| **429 Too Many Requests**     | Rate limit exceeded                                              | Sending too many RAG requests to API in short time            |
| **500 Internal Server Error** | LLM or backend failure                                           | RAG pipeline crashes during LLM call or embedding computation |
| **503 Service Unavailable**   | Temporary backend issue / overload                               | Vector DB or LLM service down temporarily                     |


## Pydantic

Pydantic is a Python library used for data validation, parsing, and serialization based on Python type hints. In FastAPI, it is used to validate incoming request data and ensure that the data matches the expected schema.

it to the required Python types, and returns a 422 Validation Error if the data is invalid. This eliminates manual validation code, improves data consistency, and also enables FastAPI to generate API documentation automatically.
```python
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class ChatRequest(BaseModel):
    question: str

@app.post("/chat")
async def chat(request: ChatRequest):
    return {"question": request.question}
```

## synchronous and asynchronous programming

Synchronous (Sync) programming executes tasks one after another. A task must finish before the next one starts.

Asynchronous (Async) programming allows a program to start another task while waiting for an I/O operation (such as a database query or API call) to complete.

## async and await in FastAPI

async and await are Python keywords used for asynchronous programming in FastAPI. The async keyword defines an asynchronous function, and await pauses execution until an asynchronous operation completes without blocking the server.

```python
from fastapi import FastAPI
import asyncio

app = FastAPI()

@app.get("/")
async def home():
    await asyncio.sleep(5)
    return {"message": "Hello"}
```

## async endpoints

I use async endpoints for I/O-bound operations, such as database queries, Redis access, Azure OpenAI calls, Azure AI Search, file I/O, and external API requests. These operations spend most of their time waiting, so async allows FastAPI to handle other requests concurrently, improving scalability and throughput. 

I avoid async for CPU-bound tasks like image processing, video processing, complex computations, or ML model training because they consume CPU resources rather than waiting. Such tasks are better executed in background workers or separate processes to avoid blocking the application.

## handle multiple concurrent

FastAPI handles multiple concurrent requests using ASGI, Uvicorn, and an event loop. When an async endpoint reaches an await statement, such as waiting for a database query or an Azure OpenAI response, it temporarily yields control back to the event loop. 

The event loop then processes other incoming requests instead of blocking. 

Once the I/O operation completes, execution resumes from where it paused. This non-blocking model allows FastAPI to efficiently handle many concurrent I/O-bound requests, making it ideal for APIs, AI applications, and RAG systems.

## Dependency Injection (Depends)

Dependency Injection (DI) is a design pattern where FastAPI automatically provides required objects or services (called dependencies) to an endpoint instead of the endpoint creating them itself.

In FastAPI, Dependency Injection is implemented using the Depends() function.

Instead of creating resources like database sessions, authentication objects, or Redis clients inside every endpoint, we define them as reusable dependencies. FastAPI resolves these dependencies before executing the endpoint and injects the returned object. This reduces code duplication, improves maintainability, and makes applications easier to test. 

In production, we commonly use Depends() for database sessions, JWT authentication, configuration, and external service clients like Redis or Azure OpenAI.

## middleware

Middleware is a component that executes before and after every HTTP request in a FastAPI application.

It sits between the client and the endpoint, allowing you to inspect, modify, or process requests and responses.

It intercepts incoming requests, can perform operations such as logging, authentication, CORS handling, request timing, or rate limiting, then forwards the request to the endpoint.

After the endpoint returns a response, middleware can modify the response or record metrics before sending it back to the client. 

In our RAG application, we use middleware for logging, request tracing, performance monitoring with Azure Monitor/Application Insights, and enforcing security-related checks.

## CORS

CORS (Cross-Origin Resource Sharing) is a browser security mechanism that controls whether a web application running on one origin can access resources from another origin.

Browsers enforce the Same-Origin Policy, so a React frontend running on localhost:3000 cannot access a FastAPI backend on localhost:8000 unless the backend explicitly allows it. 

In FastAPI, we enable CORS using the CORSMiddleware, where we configure allowed origins, HTTP methods, headers, and credentials. In production, we whitelist only trusted frontend domains instead of allowing all origins.

## Middleware vs Dependency Injection 

Middleware is used for application-wide functionality and runs for every request before and after the endpoint executes. It's commonly used for logging, CORS, request timing, security headers, and monitoring. 

Dependency Injection, using Depends(), is used for endpoint-specific functionality. FastAPI resolves the dependency before executing the endpoint and injects objects such as authenticated users, database sessions, or Redis clients. In short, Middleware is for global request/response processing, while Dependency Injection is for providing reusable services or business logic to specific endpoints.

## 422 Error 

FastAPI returns HTTP 422 Unprocessable Entity when the request format is valid, but the request data does not satisfy the validation rules defined by the Pydantic model. For example, if a required field is missing, a field has the wrong data type, or a value violates constraints, Pydantic detects the error before the endpoint executes. FastAPI then automatically returns a 422 response with detailed validation errors, helping clients identify exactly what needs to be corrected.

## Authentication and Authorization

Authentication is the process of verifying a user's identity—for example, by checking a username and password or validating a JWT token. 

Authorization determines what an authenticated user is allowed to access, such as APIs, resources, or specific actions based on roles or permissions. 

Authentication answers "Who are you?", while authorization answers "What are you allowed to do?" Authentication always occurs before authorization. In FastAPI, authentication is commonly implemented using JWT or OAuth2, while authorization is implemented by checking user roles or permissions after the token has been validated.

### Authentication Methods

OAuth 2.0 (Open Authorization) is an authorization framework that allows a user to give a third-party application limited access to their resources without sharing their username and password.

JWT (Token): JWT (JSON Web Token) is a secure, stateless authentication mechanism used to verify a user's identity. After a user logs in successfully, the server generates a JWT and sends it to the client. The client includes this token in the Authorization header for future requests..

## Upload files 

FastAPI handles file uploads using UploadFile and File. The client sends the file as multipart/form-data, and FastAPI receives it as an UploadFile object. UploadFile is preferred because it streams the file and avoids loading the entire file into memory, making it efficient for large uploads. After receiving the file, we typically validate its type and size, then save it or process it.


## Streaming Responses

StreamingResponse is a FastAPI response class that sends data to the client in small chunks instead of waiting for the entire response to be generated.

## BackGround Tasks 

Background Tasks in FastAPI allow us to execute non-critical work after sending the HTTP response to the client. FastAPI provides the BackgroundTasks class, where tasks can be added using add_task(). This improves user experience because the client receives a response immediately while operations such as sending emails, writing audit logs, or triggering notifications continue in the background. In our RAG application, background tasks can be used for lightweight post-response work, while longer-running processes like document ingestion, embedding generation, or large-scale indexing are better handled by dedicated background workers or task queues.

## FastAPI Performance in production    

To improve FastAPI performance in production, I first use async endpoints for I/O-bound operations such as database queries, Redis, Azure OpenAI, and Azure AI Search. I deploy the application using Uvicorn or Gunicorn with multiple Uvicorn workers to utilize multiple CPU cores. I implement Redis caching to reduce repeated database and LLM calls, and use connection pooling for efficient database access. Long-running tasks such as document processing or embedding generation are moved to background workers instead of blocking API requests. For AI applications, I stream responses using StreamingResponse to reduce perceived latency. I also enable GZip compression, configure rate limiting, optimize database queries, and monitor the application using Azure Monitor/Application Insights. Finally, I deploy behind a load balancer with multiple instances or Kubernetes for high availability and horizontal scalability.

## Redis caching 

Redis caching stores frequently accessed data in memory so that repeated requests can be served quickly without querying the database or calling an external API.

In FastAPI, we typically check Redis before executing expensive operations. If the data is found (cache hit), we return it immediately. Otherwise (cache miss), we fetch fresh data, store it in Redis, and return it.

### working 

I implement Redis caching in FastAPI using the cache-aside pattern. When a request arrives, I first generate a cache key and check whether the data exists in Redis. If it's a cache hit, I return the cached response immediately. 

If it's a cache miss, I fetch the data from the database or an external service such as Azure OpenAI or Azure AI Search, return the response to the client, and store it in Redis with a suitable TTL. 

In our RAG application, we cache AI responses and retrieval results to reduce latency, lower Azure OpenAI costs, and decrease the load on backend services. We also use clear cache keys, expiration times, and cache invalidation strategies to keep cached data fresh.

## Rate limiting 

restricts the number of requests a client can make within a specific time period. It protects APIs from abuse, brute-force attacks, and excessive traffic.

### Working

Rate limiting controls how many requests a client can make within a given time window to protect the API from abuse and excessive traffic. In FastAPI, I typically implement it using Redis and a middleware or dependency. 

For each request, I increment a Redis counter using a key such as rate_limit:<user_id> and set a TTL for the time window. If the request count exceeds the configured limit, the API returns HTTP 429 Too Many Requests. 

In production, I prefer enforcing rate limiting at Azure API Management (APIM) because it blocks excessive traffic before it reaches FastAPI, reducing load on backend services like Azure OpenAI, Redis, and databases.